# Data Preprocessing Pipeline
This notebook contains the complete pipeline to preprocess the landslide susceptibility dataset.

In [1]:
import os
import shutil
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from shapely.geometry import Point
from scipy.ndimage import distance_transform_edt

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
TARGET_CRS = "EPSG:32645"  # UTM Zone 45N for Sindhupalchok

# Assume the script is run from the project root
PROJECT_ROOT = Path(os.getcwd())
DATA_DIR = PROJECT_ROOT / "data"

RAW_DATA_DIR = PROJECT_ROOT / "frontend" / "Data" / "raw"
RAW_RASTER_DIR = RAW_DATA_DIR / "rasters"
RAW_VECTOR_DIR = RAW_DATA_DIR / "vectors"
RAW_INVENTORY_DIR = RAW_DATA_DIR / "inventory"

PROCESSED_DIR = PROJECT_ROOT / "frontend" / "Data" / "processed"
PROCESSED_RASTER_DIR = PROCESSED_DIR / "rasters"
PROCESSED_VECTOR_DIR = PROCESSED_DIR / "vectors"

OUTPUT_DIR = PROCESSED_DIR

# Feature definitions
RASTER_FEATURES = ["dem", "slope", "rainfall", "ndvi", "curvature"]
VECTOR_FEATURES = ["road", "river", "boundary"]

# Seed for reproducibility
RANDOM_SEED = 42

def setup_directories():
    """Ensure all required directories exist."""
    dirs = [
        RAW_RASTER_DIR, RAW_VECTOR_DIR, RAW_INVENTORY_DIR,
        PROCESSED_RASTER_DIR, PROCESSED_VECTOR_DIR, OUTPUT_DIR
    ]
    for p in dirs:
        p.mkdir(parents=True, exist_ok=True)
    print(f"Directory structure initialized in {DATA_DIR}")

# ---------------------------------------------------------
# GIS Processing Functions
# ---------------------------------------------------------
def reproject_raster(in_path: Path, out_path: Path, target_crs: str) -> Path:
    """Reproject a raster to the target CRS."""
    if not in_path.exists():
        raise FileNotFoundError(f"Missing raster: {in_path}")
    
    with rasterio.open(in_path) as src:
        src_crs = src.crs.to_string() if src.crs else None
        if src_crs and src_crs.upper() == target_crs.upper():
            out_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(in_path, out_path)
            return out_path
        
        transform, width, height = calculate_default_transform(
            src.crs, target_crs, src.width, src.height, *src.bounds
        )
        kwargs = src.meta.copy()
        kwargs.update({
            "crs": target_crs,
            "transform": transform,
            "width": width,
            "height": height,
        })
        
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(out_path, "w", **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=target_crs,
                    resampling=Resampling.bilinear,
                )
    return out_path

def reproject_vector(in_path: Path, out_path: Path, target_crs: str) -> Path:
    """Reproject a vector layer to the target CRS."""
    if not in_path.exists():
        raise FileNotFoundError(f"Missing vector: {in_path}")
    
    gdf = gpd.read_file(in_path)
    gdf = gdf.to_crs(target_crs)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    gdf.to_file(out_path)
    return out_path

def align_raster_to_ref(in_path: Path, out_path: Path, ref_path: Path, resampling: Resampling = Resampling.bilinear) -> Path:
    """Resample and align a raster to a reference raster grid (DEM)."""
    with rasterio.open(ref_path) as ref:
        ref_meta = ref.meta.copy()
    
    with rasterio.open(in_path) as src:
        kwargs = src.meta.copy()
        kwargs.update({
            "crs": ref_meta["crs"],
            "transform": ref_meta["transform"],
            "width": ref_meta["width"],
            "height": ref_meta["height"],
        })
        
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(out_path, "w", **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=ref_meta["transform"],
                    dst_crs=ref_meta["crs"],
                    resampling=resampling,
                )
    return out_path

def rasterize_distance(vector_path: Path, ref_path: Path, out_path: Path) -> Path:
    """Create a distance-to-feature raster using a reference raster grid."""
    with rasterio.open(ref_path) as ref:
        ref_meta = ref.meta.copy()
        transform = ref.transform
        height, width = ref.height, ref.width
        pixel_size = (abs(transform.a), abs(transform.e))
    
    gdf = gpd.read_file(vector_path)
    shapes = [(geom, 1) for geom in gdf.geometry if geom is not None and not geom.is_empty]
    
    # Rasterize features
    raster = features.rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype="uint8",
    )
    
    # Calculate Euclidean distance from feature pixels
    distance = distance_transform_edt(raster == 0, sampling=pixel_size).astype("float32")
    
    out_meta = ref_meta.copy()
    out_meta.update({"count": 1, "dtype": "float32", "nodata": -9999})
    
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(distance, 1)
    return out_path

def generate_random_non_landslide_points(ref_raster_path: Path, n_points: int, boundary_gdf: Optional[gpd.GeoDataFrame] = None, seed: int = RANDOM_SEED) -> gpd.GeoDataFrame:
    """Generate random non-landslide points within the valid areas of the reference raster and optionally inside a boundary."""
    rng = np.random.default_rng(seed)
    
    with rasterio.open(ref_raster_path) as src:
        bounds = src.bounds
        minx, miny, maxx, maxy = bounds.left, bounds.bottom, bounds.right, bounds.top
        crs = src.crs
        
        # Read the mask to ensure we only sample where DEM is valid
        data = src.read(1, masked=True)
        valid_mask = ~data.mask
        
    # If boundary is provided, ensure its CRS matches the raster CRS
    if boundary_gdf is not None:
        boundary_gdf = boundary_gdf.to_crs(crs)
        # Use a single geometry for fast containment checking
        boundary_geom = boundary_gdf.union_all()
    else:
        boundary_geom = None
    
    points = []
    while len(points) < n_points:
        x = rng.uniform(minx, maxx)
        y = rng.uniform(miny, maxy)
        pt = Point(x, y)
        
        # Check boundary containment first (faster if boundary is smaller than raster bounds)
        if boundary_geom is not None and not boundary_geom.contains(pt):
            continue
            
        with rasterio.open(ref_raster_path) as src:
            # Check if point falls in valid data area
            row, col = src.index(x, y)
            if 0 <= row < src.height and 0 <= col < src.width:
                if valid_mask[row, col]:
                    points.append(pt)
                    
    return gpd.GeoDataFrame({"geometry": points}, crs=crs)

def extract_raster_values(raster_paths: Dict[str, Path], points_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """Extract values from multiple rasters at given point locations."""
    coords = []
    for geom in points_gdf.geometry:
        if geom is None or geom.is_empty:
            coords.append((float("nan"), float("nan")))
        else:
            # Use centroid in case geometry is not a point
            c = geom.centroid
            coords.append((float(c.x), float(c.y)))
    
    extracted_data = {}
    for feature_name, path in raster_paths.items():
        if not path.exists():
            print(f"Warning: Raster for {feature_name} not found. Returning NaN.")
            extracted_data[feature_name] = [np.nan] * len(coords)
            continue
            
        with rasterio.open(path) as src:
            samples = [s[0] for s in src.sample(coords)]
            # Replace nodata with NaN
            nodata = src.nodata
            if nodata is not None:
                samples = [np.nan if s == nodata else s for s in samples]
            extracted_data[feature_name] = samples
            
    return pd.DataFrame(extracted_data)

# ---------------------------------------------------------
# Main Pipeline
# ---------------------------------------------------------
def main():
    setup_directories()
    print("Starting Data Preprocessing Pipeline...")
    
    # 1. Discover Input Files
    # (Assuming user places files with matching prefixes in raw folders)
    def find_file(directory: Path, keywords: List[str], ext: str) -> Optional[Path]:
        for p in directory.glob(f"*{ext}"):
            if any(k.lower() in p.name.lower() for k in keywords):
                return p
        return None

    raster_inputs = {
        "dem": find_file(RAW_RASTER_DIR, ["dem", "elevation"], ".tif"),
        "slope": find_file(RAW_RASTER_DIR, ["slope"], ".tif"),
        "rainfall": find_file(RAW_RASTER_DIR, ["rain", "precip"], ".tif"),
        "ndvi": find_file(RAW_RASTER_DIR, ["ndvi"], ".tif"),
        "curvature": find_file(RAW_RASTER_DIR, ["curv"], ".tif"),
    }
    
    vector_inputs = {
        "road": find_file(RAW_VECTOR_DIR, ["road"], ".shp"),
        "river": find_file(RAW_VECTOR_DIR, ["river", "stream"], ".shp"),
        "boundary": find_file(RAW_VECTOR_DIR, ["boundary", "district"], ".shp"),
    }
    
    inventory_path = find_file(RAW_INVENTORY_DIR, ["landslide", "inventory"], ".csv")
    
    # Validate core inputs
    if not raster_inputs["dem"]:
        print("ERROR: DEM raster is required. Please place it in data/raw/rasters/")
        return
    if not inventory_path:
        print("ERROR: Landslide inventory CSV is required. Please place it in data/raw/inventory/")
        return

    # 2. Reproject and Align Rasters
    print("\n--- Processing Rasters ---")
    aligned_rasters = {}
    
    # Process DEM first to serve as reference
    dem_reproj = PROCESSED_RASTER_DIR / "dem_reproj.tif"
    print(f"Reprojecting DEM to {TARGET_CRS}...")
    dem_ref = reproject_raster(raster_inputs["dem"], dem_reproj, TARGET_CRS)
    aligned_rasters["dem"] = dem_ref
    
    # Process other rasters
    for feat in ["slope", "rainfall", "ndvi", "curvature"]:
        if raster_inputs[feat]:
            print(f"Reprojecting and aligning {feat}...")
            reproj_path = PROCESSED_RASTER_DIR / f"{feat}_reproj.tif"
            aligned_path = PROCESSED_RASTER_DIR / f"{feat}_aligned.tif"
            
            # Reproject first, then align to DEM grid
            reproject_raster(raster_inputs[feat], reproj_path, TARGET_CRS)
            align_raster_to_ref(reproj_path, aligned_path, dem_ref)
            aligned_rasters[feat] = aligned_path
            
            # Clean up intermediate file
            reproj_path.unlink()
        else:
            print(f"Warning: {feat} raster not found. It will be excluded from the dataset.")

    # 3. Process Vectors and Create Distance Rasters
    print("\n--- Processing Vectors ---")
    boundary_gdf_utm = None
    if vector_inputs["boundary"]:
        print(f"Reprojecting boundary vector...")
        reproj_bnd = PROCESSED_VECTOR_DIR / "boundary_reproj.shp"
        reproject_vector(vector_inputs["boundary"], reproj_bnd, TARGET_CRS)
        boundary_gdf_utm = gpd.read_file(reproj_bnd)
        
    for feat in ["road", "river"]:
        if vector_inputs[feat]:
            print(f"Reprojecting and rasterizing distance for {feat}...")
            reproj_path = PROCESSED_VECTOR_DIR / f"{feat}_reproj.shp"
            dist_path = PROCESSED_RASTER_DIR / f"distance_to_{feat}.tif"
            
            reproject_vector(vector_inputs[feat], reproj_path, TARGET_CRS)
            rasterize_distance(reproj_path, dem_ref, dist_path)
            aligned_rasters[f"distance_to_{feat}"] = dist_path
        else:
            print(f"Warning: {feat} vector not found. Distance feature will be excluded.")

    # 4. Process Inventory and Feature Extraction
    print("\n--- Extracting Features ---")
    # Load inventory
    df_inv = pd.read_csv(inventory_path)
    
    # Determine lat/lon column names dynamically
    lower_cols = {c.lower(): c for c in df_inv.columns}
    lat_col = next((lower_cols[c] for c in lower_cols if "lat" in c or "y" in c), None)
    lon_col = next((lower_cols[c] for c in lower_cols if "lon" in c or "x" in c), None)
    
    if not lat_col or not lon_col:
        print("ERROR: Could not find latitude/longitude columns in inventory CSV.")
        return
        
    df_inv = df_inv.dropna(subset=[lat_col, lon_col]).copy()
    
    # Convert inventory to GeoDataFrame and reproject to target CRS
    # Assuming the input CSV coordinates are in EPSG:4326 (standard lat/lon)
    inv_gdf = gpd.GeoDataFrame(
        df_inv, 
        geometry=gpd.points_from_xy(df_inv[lon_col], df_inv[lat_col]),
        crs="EPSG:4326"
    )
    inv_gdf = inv_gdf.to_crs(TARGET_CRS)
    inv_gdf["label"] = 1  # Landslide presence
    
    # Generate negative samples (1:1 ratio)
    n_positive = len(inv_gdf)
    print(f"Found {n_positive} positive landslide samples.")
    print("Generating non-landslide (negative) samples for class balance...")
    neg_gdf = generate_random_non_landslide_points(dem_ref, n_points=n_positive, boundary_gdf=boundary_gdf_utm)
    neg_gdf["label"] = 0
    
    # Combine samples
    samples_gdf = pd.concat([inv_gdf[["geometry", "label"]], neg_gdf], ignore_index=True)
    
    # Extract values from all aligned rasters
    print("Extracting raster values at sample points...")
    features_df = extract_raster_values(aligned_rasters, samples_gdf)
    
    # 5. Build Final Dataset and Clean Missing Values
    print("\n--- Finalizing Dataset ---")
    
    # Re-extract coordinates in Target CRS (UTM) or fallback to WGS84 depending on need.
    # For ML we typically keep the raw UTM coordinates or the original Lat/Lon.
    # Let's save original Lat/Lon for easier plotting later.
    samples_wgs84 = samples_gdf.to_crs("EPSG:4326")
    features_df["latitude"] = samples_wgs84.geometry.y
    features_df["longitude"] = samples_wgs84.geometry.x
    features_df["label"] = samples_gdf["label"].values
    
    # Handle Missing Values
    # Impute numerical columns with median
    feature_cols = [c for c in features_df.columns if c not in ["latitude", "longitude", "label"]]
    
    missing_counts = features_df[feature_cols].isna().sum()
    if missing_counts.sum() > 0:
        print("Handling missing values (median imputation):")
        print(missing_counts[missing_counts > 0])
        features_df[feature_cols] = features_df[feature_cols].fillna(features_df[feature_cols].median())
    
    # Drop rows that somehow still have NaNs (e.g. out of bounds)
    initial_len = len(features_df)
    features_df = features_df.dropna()
    dropped = initial_len - len(features_df)
    if dropped > 0:
        print(f"Dropped {dropped} rows that were entirely invalid (e.g., outside study area).")
        
    # Reorder columns: lat, lon, features..., label
    final_cols = ["latitude", "longitude"] + feature_cols + ["label"]
    features_df = features_df[final_cols]
    
    # Enforce snake_case for all column names
    import re
    def to_snake_case(name):
        name = re.sub(r'(?<!^)(?=[A-Z])', '_', str(name))
        name = re.sub(r'[^a-zA-Z0-9_]', '_', name)
        return name.lower().strip('_')
        
    features_df.columns = [to_snake_case(c) for c in final_cols]
    
    # 6. Export
    out_csv = OUTPUT_DIR / "ml_ready_dataset.csv"
    features_df.to_csv(out_csv, index=False)
    print(f"\nSUCCESS! Clean dataset saved to: {out_csv}")
    print("\n--- Final Class Balance ---")
    print(f"Landslide (1):     {len(features_df[features_df['label'] == 1])}")
    print(f"Non-landslide (0): {len(features_df[features_df['label'] == 0])}")
    print("-" * 27)
    print(f"Total samples:     {len(features_df)}\n")

if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'numpy'